# 001 — Market data loader

Demonstrates `alphatools.marketdata.load_bars`: loading daily and intraday bars,
reusing the local parquet cache, and a tiny SPY-vs-RSP return divergence.

> Requires network access on first run (it fetches from yfinance and caches the
> result). Re-running reads from the local cache.

In [1]:
import polars as pl

from alphatools.marketdata import load_bars

## 1. Load SPY / RSP / QQQ daily bars for 2024

In [2]:
bars = load_bars(
    symbols=["SPY", "RSP", "QQQ"],
    start="2024-01-01",
    end="2025-01-01",
    interval="1d",
)
print(bars.shape)
bars.head()

(756, 10)


symbol,timestamp,open,high,low,close,volume,source,interval,adjusted
str,datetime[μs],f64,f64,f64,f64,i64,str,str,bool
"""QQQ""",2024-01-02 00:00:00,400.901154,401.148111,395.369296,397.690704,58026900,"""yfinance""","""1d""",true
"""QQQ""",2024-01-03 00:00:00,395.06311,396.120096,393.047957,393.482574,47002800,"""yfinance""","""1d""",true
"""QQQ""",2024-01-04 00:00:00,391.615607,394.727267,391.240226,391.45755,39432800,"""yfinance""","""1d""",true
"""QQQ""",2024-01-05 00:00:00,391.625446,394.697585,390.528939,391.921783,44922800,"""yfinance""","""1d""",true
"""QQQ""",2024-01-08 00:00:00,393.146741,400.308513,392.998572,400.022064,42473800,"""yfinance""","""1d""",true


## 2. Inspect the canonical schema

In [3]:
bars.schema

Schema([('symbol', String),
        ('timestamp', Datetime(time_unit='us', time_zone=None)),
        ('open', Float64),
        ('high', Float64),
        ('low', Float64),
        ('close', Float64),
        ('volume', Int64),
        ('source', String),
        ('interval', String),
        ('adjusted', Boolean)])

## 3. Re-load to demonstrate cache reuse

The second call reads parquet from `./.alphatools/cache/marketdata/` instead of
hitting yfinance, so it returns quickly and works offline.

In [4]:
bars_cached = load_bars(["SPY", "RSP", "QQQ"], "2024-01-01", "2025-01-01")
assert bars_cached.equals(bars)
print("cache reuse OK", bars_cached.shape)

cache reuse OK (756, 10)


## 4. Load 5m bars for a small recent range

yfinance only serves intraday history for a short recent window, so pick a range
within the last ~60 days. `source_only` avoids caching a partial intraday year.

In [5]:
from datetime import date, timedelta

today = date.today()
intraday = load_bars(
    ["SPY"],
    start=today - timedelta(days=5),
    end=today + timedelta(days=1),
    interval="5m",
    mode="source_only",
)
print(intraday.shape)
intraday.head()

(312, 10)


symbol,timestamp,open,high,low,close,volume,source,interval,adjusted
str,datetime[μs],f64,f64,f64,f64,i64,str,str,bool
"""SPY""",2026-05-26 09:30:00,750.01001,750.440002,749.359985,749.429993,2038420,"""yfinance""","""5m""",true
"""SPY""",2026-05-26 09:35:00,749.44989,750.0,749.130981,749.72998,659614,"""yfinance""","""5m""",true
"""SPY""",2026-05-26 09:40:00,749.719971,750.700012,749.686584,750.630005,522602,"""yfinance""","""5m""",true
"""SPY""",2026-05-26 09:45:00,750.650024,751.119995,750.640015,750.97998,600452,"""yfinance""","""5m""",true
"""SPY""",2026-05-26 09:50:00,750.97998,751.039978,749.580017,750.140015,644375,"""yfinance""","""5m""",true


## 5. Build a wide close-price frame

In [6]:
wide = bars.pivot(values="close", index="timestamp", on="symbol").sort("timestamp")
wide.head()

timestamp,QQQ,RSP,SPY
datetime[μs],f64,f64,f64
2024-01-02 00:00:00,397.690704,152.109467,459.991211
2024-01-03 00:00:00,393.482574,149.864212,456.234619
2024-01-04 00:00:00,391.45755,149.613663,454.765015
2024-01-05 00:00:00,391.921783,150.047302,455.387878
2024-01-08 00:00:00,400.022064,151.656555,461.888977


## 6. Simple daily returns for SPY and RSP

In [7]:
returns = wide.select(
    "timestamp",
    (pl.col("SPY").pct_change()).alias("spy_ret"),
    (pl.col("RSP").pct_change()).alias("rsp_ret"),
).drop_nulls()
returns.head()

timestamp,spy_ret,rsp_ret
datetime[μs],f64,f64
2024-01-03 00:00:00,-0.008167,-0.014761
2024-01-04 00:00:00,-0.003221,-0.001672
2024-01-05 00:00:00,0.00137,0.002898
2024-01-08 00:00:00,0.014276,0.010725
2024-01-09 00:00:00,-0.001517,-0.005083


## 7. SPY-minus-RSP return divergence

A quick proxy for cap-weighted vs equal-weighted S&P 500 dispersion.

In [8]:
divergence = returns.with_columns((pl.col("spy_ret") - pl.col("rsp_ret")).alias("spy_minus_rsp"))
divergence = divergence.with_columns(
    pl.col("spy_minus_rsp").cum_sum().alias("cumulative_divergence")
)
divergence.select("timestamp", "spy_minus_rsp", "cumulative_divergence").tail()

timestamp,spy_minus_rsp,cumulative_divergence
datetime[μs],f64,f64
2024-12-24 00:00:00,0.00357,0.121237
2024-12-26 00:00:00,-0.001566,0.119671
2024-12-27 00:00:00,-0.003725,0.115946
2024-12-30 00:00:00,-0.001337,0.114608
2024-12-31 00:00:00,-0.005468,0.109141


In [9]:
print("mean daily SPY-RSP divergence:", divergence["spy_minus_rsp"].mean())
print("cumulative SPY-RSP divergence over 2024:", divergence["cumulative_divergence"][-1])

mean daily SPY-RSP divergence: 0.00043482283233202466
cumulative SPY-RSP divergence over 2024: 0.1091405309153382
